# 12.5 APCS 排序實戰應用：區間線段排序、貪婪前導與已排序雙指標

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_12-5_apcs_sorting_patterns_intervals_and_pointers.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**先備知識**：已掌握 Chapter 12 前四節之所有排序語法、`reverse=True`、`key=` 參數與 `lambda` 匿名函數。

---

### 學習導覽：從語法到戰術——先排序，混亂立刻變規律！

在學完了 `sort()`、`sorted()`、具名函數以及 `lambda` 多準則排序之後，很多學習者常常會產生一個疑問：
「老師，我已經知道怎麼把數字或字串排好序了，但在 APCS 競賽實作題中，排序到底能幫我們解決什麼樣的高難度問題？」

在演算法的世界裡，有一句傳誦百年的至理名言：**「先排序，混亂立刻變規律！」**
面對雜亂無章的輸入資料，如果你直接用暴力迴圈硬碰硬，往往會落入 $O(N^2)$ 的超時深淵（Time Limit Exceeded, TLE）。
但是，只要花費 $O(N \log N)$ 的代價將資料「按某種關鍵規則排好序」，原本看似無從下手的棘手難題，就會瞬間變成一條清晰可見的康莊大道：
- **區間重疊與線段合併**：依起點排序後，線段由左向右推進，重疊情況一目了然！
- **貪婪活動選擇**：依結束時間排序，永遠挑選最早結束的活動，輕鬆排入最多行程！
- **極速相鄰比對**：排序後最接近的數值必定緊緊相鄰，$O(N \log N)$ 秒殺全數列最小差值！
- **已排序相向雙指標**：頭尾兩根指標向中間夾擊，線性 $O(N)$ 破解經典 Two Sum 兩數之和！

在本單元中，我們將透過 6 個平緩的微階梯，打通排序與高階演算法之間的任督二脈：
1. **12.5.1 APCS 解題心法**：「先排序，混亂立刻變規律！」（由無序邁向有序）。
2. **12.5.2 區間線段依起點排序**：`(start, end)`（線段覆蓋長度、重疊檢查前導）。
3. **12.5.3 區間依結束時間排序**：活動安排問題（貪婪選擇最多不衝突活動原型）。
4. **12.5.4 排序後相鄰元素比對**：極速找出重複元素或最小相鄰差值（$O(N \log N)$ 取代 $O(N^2)$）。
5. **12.5.5 已排序序列的雙指標夾擊（Two Pointers）**：左右相向極速 $O(N)$ 解 Two Sum。
6. **12.5.6 競賽時空權衡**：何時值得花 $O(N \log N)$ 排序換取後續的高速線性處理？

讓我們跨出單純的語法學習，真正像一位頂尖選手一樣思考演算法！

### 12.5.1 APCS 解題心法：「先排序，混亂立刻變規律！」（由無序邁向有序）

#### 1. 生活故事比喻：散落一地的撲克牌與整理書架
想像有人將一副 52 張撲克牌隨手灑在客廳地板上，雜亂無章。現在對方問你：「請問這堆牌裡面有沒有重複的黑桃 7？哪兩張牌的點數差距最小？」
如果你就這樣趴在地上，一張一張互相對照，你的眼睛會看花、頭昏腦脹，需要進行高達數千次的比對（這就是暴力法 $O(N^2)$ 的真實寫照）。
但一個聰明的人會怎麼做？
他會先把整堆撲克牌撿起來，花 10 秒鐘將它們依照花色與點數「由小到大排整齊」拿在手裡。
一旦排好序，奇蹟發生了：
- 有沒有重複的牌？只要看有沒有相鄰兩張一模一樣即可！
- 哪兩張差值最小？差距最小的牌一定緊緊貼在一起！
這就是**「先排序，混亂立刻變規律」**的無窮魅力！

#### 2. 底層運作機制：為無序宇宙建立單調性（Monotonicity）
在數學與電腦科學中，「無序（Unordered）」意味著任何一個元素都可能出現在任何位置，演算法無法做出任何前瞻性的預測，只能全盤盲目搜尋。
而「有序（Ordered）」則賦予了序列強大的**單調性（Monotonicity）**：
- 如果 $i < j$，則必定有 $a[i] \le a[j]$。
- 這意味著：在位置 $i$ 後面的所有數字，保證通通都大於等於 $a[i]$！
這種單調性讓我們擁有了「提早煞車（Early Break）」、「二分折半（Binary Search）」以及「指針單向滑動（Two Pointers）」的超能力！

#### 3. 初學者常見陷阱：面對大數據盲目寫雙重迴圈
許多初學者拿到題目，第一直覺就是寫雙重迴圈 `for i in range(n): for j in range(i + 1, n): ...`。
當題目資料量 $N = 1,000$ 時，雙重迴圈大約運算一百萬次，電腦還勉強跑得動；但當 APCS 題目的測資放大到 $N = 100,000$ 時，$N^2 = 10^{10}$（一百億次運算），程式必定超時崩潰（TLE）。
此時如果「先花 $O(N \log N)$ 排序」，再用「單重迴圈 $O(N)$ 線性掃描」，總運算量瞬間降至幾百萬次，0.1 秒輕鬆過關！

#### 4. APCS 實戰視野
在 APCS 評測系統中，幾乎所有看似需要複雜暴力搜尋的題目，只要題意中出現「挑選最佳組合」、「檢查區間重疊」、「尋找最接近兩數」，你的腦海中第一個要跳出的靈感火花，必定是「能否先對資料進行排序？」！

In [ ]:
# 範例 12.5.1：排序前 vs 排序後的資訊規律對比
# 原始雜亂數列
nums = [47, 12, 89, 23, 99, 56, 12, 78]
print("原始無序數列：", nums)

# 暴力思維：不知道最大最小值在哪裡，不知道是否有重複，需要全盤搜尋

# 啟動排序：由混亂走向有序
nums.sort()
print("排序後有序數列：", nums)

# 規律 1：極值瞬間出現在邊界
print("極速取得最小值（首位）：", nums[0])
print("極速取得最大值（末位）：", nums[-1])

# 規律 2：相同元素必定相鄰！極速檢查是否存在重複值
has_duplicate = False
for i in range(len(nums) - 1):
    if nums[i] == nums[i + 1]:
        print(f"發現相鄰重複元素：{nums[i]}")
        has_duplicate = True
        break
print("是否存在重複元素：", has_duplicate)

In [ ]:
# 填空題 12.5.1：排序後極速找出數列中位數（Median）
# 任務：將長度為奇數的數列排序後，取出最正中間的中位數。
raw_scores = [75, 90, 62, 88, 79]

# 請先將 raw_scores 進行升序排序
sorted_scores = ___(raw_scores)

# 中間索引為 len // 2
mid_idx = len(sorted_scores) // 2
median = sorted_scores[___]

print("排序後的成績隊列：", sorted_scores)
print("中位數分數為：", median)

In [ ]:
# ==========================================
# [4] Code 練習題 12.5.1
# 任務說明：
# 給定一組整數清單 data。
# 請撰寫程式：
# 1. 將 data 進行升序排序。
# 2. 透過單重迴圈（檢查相鄰元素），判斷清單中是否「所有相鄰元素之間的差值都恰好為 1」（即連續整數列）。
# 3. 若是印出 "是連續整數"，否則印出 "不是連續整數"。
#
# 【公開測試資料 1】
# data = [5, 2, 4, 1, 3]
# 預期輸出：
# 排序後： [1, 2, 3, 4, 5]
# 檢驗結果： 是連續整數
#
# 【公開測試資料 2】
# data = [10, 12, 11, 14]
# 預期輸出：
# 排序後： [10, 11, 12, 14]
# 檢驗結果： 不是連續整數
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
data = [5, 2, 4, 1, 3]
data.sort()
print("排序後：", data)
is_consecutive = True
for i in range(len(data) - 1):
    if data[i + 1] - data[i] != 1:
        is_consecutive = False
        break
if is_consecutive:
    print("檢驗結果： 是連續整數")
else:
    print("檢驗結果： 不是連續整數")

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.5.1
# 任務說明：
# 某遊戲公測排行榜給定玩家等級清單：levels = [15, 42, 8, 23, 99, 15, 60, 42]
# 請寫出一段程式：
# 1. 將等級排序。
# 2. 透過一次線性走訪（比對相鄰），收集所有「重複出現的等級」放入 duplicates 清單中（重複的不需重複記錄）。
# 3. 印出所有出現重複的等級。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
levels = [15, 42, 8, 23, 99, 15, 60, 42]
levels.sort()
duplicates = []
for i in range(len(levels) - 1):
    if levels[i] == levels[i + 1]:
        if not duplicates or duplicates[-1] != levels[i]:
            duplicates.append(levels[i])
print("排序後等級：", levels)
print("重複出現的等級：", duplicates)

### 12.5.2 區間線段（Intervals）依起點排序：`(start, end)`（線段覆蓋長度、重疊檢查前導）

#### 1. 生活故事比喻：數線上的火車軌道鋪設
想像工程隊要在鐵路數線上鋪設枕木，每根枕木都是一條線段，記錄為 `[起點, 終點]`，例如 `[1, 4]`、`[8, 10]`、`[3, 6]`。
如果這些枕木像抓周一樣隨意丟在地上，你要如何判斷「枕木之間有沒有互相重疊擦撞」？
如果你隨意拿兩根對照，會算到發瘋。
但如果我們做一件事：**將所有枕木依照「起點由西向東（由小到大）」排成一列**！
當你沿著鐵路從左往右走時：
只要下一根枕木的起點，比當前枕木的最遠終點還要小（`next_start < current_end`），你就百分之百確定——**這兩根枕木必定發生了重疊**！

#### 2. 底層運作機制：線段樹掃描線（Sweep-line）的原型
在 APCS 與各類演算法題庫中，「區間線段（Intervals）」是最經典的二維幾何結構。
標準的線段排序寫法：
```python
intervals.sort(key=lambda seg: seg[0])  # 依起點升序排列
```
排好序後，我們維護一個「當前已知覆蓋的最遠終點 `max_end`」：
- 檢查下一個線段 `[cur_start, cur_end]`：
  - 若 `cur_start < max_end`：代表當前線段與前方的覆蓋範圍**發生了重疊（Overlap）**！
  - 若 `cur_start >= max_end`：代表中間出現了空隙，這是一個完全獨立的全新線段！

#### 3. 初學者常見陷阱：起點相同時的次要條件
當多條線段的起點相同時（例如 `[2, 5]` 與 `[2, 9]`），哪一條應該排在前面？
通常若起點相同，我們會希望覆蓋範圍更長的排在前面（即終點降序），或者終點升序。
初學同學若只寫 `key=lambda x: x[0]`，則起點相同時會維持原始不確定的順序。若題目要求嚴格，記得寫成雙條件：`key=lambda x: (x[0], x[1])`。

#### 4. APCS 實戰視野
著名的 APCS 經典實作題（如線段覆蓋長度、區間聯集計算），其核心第一步 100% 都是「依照起點升序排序」！排序將二維散亂的區間拉平成了一維時間軸的有序推進，是打通區間演算法的第一把金鑰匙。

In [ ]:
# 範例 12.5.2：區間線段依起點排序與重疊檢查
# 一組雜亂的區間線段：[起點, 終點]
intervals = [[8, 12], [2, 5], [4, 7], [15, 20], [1, 3]]
print("原始線段清單：", intervals)

# 第一步：依「起點（索引 0）」升序排序
intervals.sort(key=lambda seg: seg[0])
print("依起點排序後線段：", intervals)

# 第二步：由左向右推進，檢查相鄰線段是否有重疊
print("\n逐一檢查相鄰線段重疊情況：")
for i in range(len(intervals) - 1):
    cur_seg = intervals[i]
    next_seg = intervals[i + 1]
    # 若下一段的起點小於當前段的終點，代表兩者重疊！
    if next_seg[0] < cur_seg[1]:
        print(f"  線段 {cur_seg} 與 {next_seg} 重疊！")
    else:
        print(f"  線段 {cur_seg} 與 {next_seg} 無重疊（互斥分離）")

In [ ]:
# 填空題 12.5.2：合併重疊線段原型
# 任務：將相鄰重疊的兩線段合併為一條大線段。
# 若線段 A [s1, e1] 與線段 B [s2, e2] 重疊（s2 <= e1），合併後的新終點為 max(e1, e2)。
segA = [2, 6]
segB = [4, 9]

# 檢查 segB 的起點是否小於等於 segA 的終點
is_overlap = segB[0] <= segA[___]

if is_overlap:
    # 合併後起點不變，終點取兩者最大值
    merged_seg = [segA[0], max(segA[1], segB[___])]
    print("兩線段合併結果：", merged_seg)

In [ ]:
# ==========================================
# [4] Code 練習題 12.5.2
# 任務說明：
# 給定一批排程區間 intervals，每筆為 [開始時間, 結束時間]。
# 請撰寫程式：
# 1. 依開始時間（起點）升序排序。
# 2. 檢查整份清單中，是否存在「任何一對互相重疊的排程」（相鄰檢查即可）。
# 3. 若有任何重疊印出 "存在重疊衝突"，若完全無重疊則印出 "所有排程皆順暢無衝突"。
# （注意：若前一段結束時間等於後一段開始時間，如 [1, 3] 與 [3, 5]，不算重疊）
#
# 【公開測試資料 1】
# intervals = [[9, 11], [14, 16], [10, 12]]
# 預期輸出：
# 排序後： [[9, 11], [10, 12], [14, 16]]
# 檢查結果： 存在重疊衝突
#
# 【公開測試資料 2】
# intervals = [[1, 3], [5, 8], [3, 5]]
# 預期輸出：
# 排序後： [[1, 3], [3, 5], [5, 8]]
# 檢查結果： 所有排程皆順暢無衝突
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
intervals = [[9, 11], [14, 16], [10, 12]]
intervals.sort(key=lambda seg: seg[0])
print("排序後：", intervals)
has_conflict = False
for i in range(len(intervals) - 1):
    if intervals[i + 1][0] < intervals[i][1]:
        has_conflict = True
        break
if has_conflict:
    print("檢查結果： 存在重疊衝突")
else:
    print("檢查結果： 所有排程皆順暢無衝突")

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.5.2
# 任務說明：
# 某自習室有一批同學的預約時段：
# bookings = [[13, 15], [9, 11], [14, 17], [10, 12], [18, 20]]
# 請將預約時段依照開始時間排序後，
# 計算這些時段在數線上「被預約的總時長（重疊部分不得重複計算！）」。
# （提示：維護當前覆蓋的 start 與 end，遇到重疊則延伸 end = max(end, cur_end)；遇到斷開則累加長度）
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
bookings = [[13, 15], [9, 11], [14, 17], [10, 12], [18, 20]]
bookings.sort(key=lambda b: b[0])

total_hours = 0
cur_start, cur_end = bookings[0]

for s, e in bookings[1:]:
    if s <= cur_end:
        cur_end = max(cur_end, e)
    else:
        total_hours += (cur_end - cur_start)
        cur_start, cur_end = s, e
total_hours += (cur_end - cur_start)

print("排序後預約時段：", bookings)
print("自習室實際被佔用總時長：", total_hours, "小時")

### 12.5.3 區間依結束時間排序：活動安排問題（貪婪選擇最多不衝突活動原型）

#### 1. 生活故事比喻：會議室租借的最大效益
想像你是某大學國際會議中心的管理員，今天有 10 個不同的學生社團申請使用同一間大禮堂，每間社團的活動時間不同，如 `[9:00, 12:00]`、`[10:00, 10:30]`、`[11:00, 15:00]`。
身為管理員，你希望「在大禮堂中舉辦盡可能多場的活動（活動場次最大化）」。
你該如何挑選？
直覺上，有人會想：「先挑最早開始的活動」，但如果最早開始的活動是一場長達 8 小時的超級大演講，挑了它之後，今天整天其他活動通通辦不成了！
數學家給出了一個無比神奇且被證明為最佳的**貪婪策略（Greedy Choice）**：
**「永遠挑選『最早結束（結束時間最早）』且與前一場不衝突的活動！」**
因為它越早結束，留給後面的剩餘空閒時間就越多！

#### 2. 底層運作機制：活動選擇問題（Interval Scheduling）兩步曲
這個名震天下的演算法，第一步正是關鍵的排序：
1. **依結束時間（`end`）由小到大排序**：
   ```python
   activities.sort(key=lambda act: act[1])
   ```
2. **貪婪線性掃描**：
   - 第一個活動因為結束時間最早，必定直接入選！記錄其結束時間 `last_end = activities[0][1]`。
   - 接著依序檢視後續活動：只要某個活動的 `start >= last_end`（不衝突），就毫不猶豫將其納入，並將 `last_end` 更新為該活動的結束時間！

#### 3. 初學者常見陷阱：誤將起點當作貪婪排序標準
初學者最常見的直覺錯誤，就是習慣性地依照「起點 `start`」排序。
請記住區間兩大流派的關鍵分工：
- **檢查重疊、線段合併、覆蓋長度** ➔ **依「起點 `start`」排序**！
- **不衝突活動排程、挑選最多場次** ➔ **依「結束時間 `end`」排序**！

#### 4. APCS 實戰視野
「活動選擇問題（Activity Selection Problem）」是 APCS 實作三級與貪婪演算法的必考基石！它向我們展示了：僅僅是改變了排序的 key（從起點換成終點），原本看似需要指數級窮舉的複雜決策，竟然能在一瞬間退化為 $O(N)$ 的輕鬆線性挑選！

In [ ]:
# 範例 12.5.3：貪婪活動選擇演算法（依結束時間排序）
# 候選活動清單：(活動名稱, 開始時間, 結束時間)
activities = [
    ("社團大會", 1, 4),
    ("短片賞析", 3, 5),
    ("程式工作坊", 0, 6),
    ("吉他彈唱", 5, 7),
    ("桌遊同樂", 3, 9),
    ("魔術表演", 5, 9),
    ("晚會彩排", 6, 10),
    ("星空夜談", 8, 11)
]

# 關鍵第一步：依照「結束時間（索引 2）」由小到大升序排列！
activities.sort(key=lambda act: act[2])
print("依照結束時間排序後的活動清單：")
for act in activities:
    print(f"  {act[0]:<6}: {act[1]:2d} 點 ~ {act[2]:2d} 點")

# 關鍵第二步：貪婪挑選不衝突活動
selected = []
last_end_time = -1

for act in activities:
    name, start, end = act
    # 只要當前活動的開始時間，大於等於上一場的結束時間，就代表無衝突可排入！
    if start >= last_end_time:
        selected.append(act)
        last_end_time = end

print("\n最終貪婪挑選出的最多場次活動組合（共", len(selected), "場）：")
for act in selected:
    print(f"  ✅ {act[0]:<6} ({act[1]} 點 ~ {act[2]} 點)")

In [ ]:
# 填空題 12.5.3：貪婪挑選最多場演講
# 任務：每場演講為 [start, end]，請依 end 排序並計算最多可聽幾場演講。
lectures = [[1, 3], [2, 5], [4, 7], [1, 8], [6, 9]]

# 請將 lectures 依照結束時間（索引 1）排序
lectures.sort(key=lambda x: x[___])

count = 0
last_end = 0

for start, end in lectures:
    # 若開始時間大於等於前一場結束時間
    if start >= ___:
        count += 1
        last_end = end

print("最多可參加的演講場數：", count)

In [ ]:
# ==========================================
# [4] Code 練習題 12.5.3
# 任務說明：
# 某排練室有 5 個表演團體申請排練：(團名, 開始小時, 結束小時)。
# 請撰寫程式：
# 1. 依結束時間排序。
# 2. 貪婪挑選最多可容納的表演團體。
# 3. 印出入選的表演團體名稱清單。
#
# 【公開測試資料 1】
# requests = [("樂團A", 10, 12), ("舞團B", 11, 13), ("劇團C", 12, 14), ("魔術D", 14, 15)]
# 預期輸出：
# 入選團體： ['樂團A', '劇團C', '魔術D']
#
# 【公開測試資料 2】
# requests = [("T1", 9, 18), ("T2", 9, 10), ("T3", 10, 11)]
# 預期輸出：
# 入選團體： ['T2', 'T3']
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
requests = [("樂團A", 10, 12), ("舞團B", 11, 13), ("劇團C", 12, 14), ("魔術D", 14, 15)]
requests.sort(key=lambda r: r[2])
chosen = []
last_end = -1
for name, s, e in requests:
    if s >= last_end:
        chosen.append(name)
        last_end = e
print("入選團體：", chosen)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.5.3
# 任務說明：
# 某電視台有 6 部動畫影集待播，時段為：
# shows = [("動畫甲", 8, 11), ("動畫乙", 10, 12), ("動畫丙", 11, 14), ("動畫丁", 13, 15), ("動畫戊", 14, 17), ("動畫己", 16, 18)]
# 請寫出程式，依貪婪策略挑選最多可完整收看的不重疊動畫清單，
# 並計算所有入選動畫的「總播映時數（所有入選影集的結束時間 - 開始時間之總和）」。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
shows = [("動畫甲", 8, 11), ("動畫乙", 10, 12), ("動畫丙", 11, 14), ("動畫丁", 13, 15), ("動畫戊", 14, 17), ("動畫己", 16, 18)]
shows.sort(key=lambda s: s[2])
chosen_shows = []
last_end = -1
for name, s, e in shows:
    if s >= last_end:
        chosen_shows.append((name, s, e))
        last_end = e

total_duration = sum(e - s for name, s, e in chosen_shows)
print("入選動畫清單：", [x[0] for x in chosen_shows])
print(f"總觀看時數: {total_duration} 小時")

### 12.5.4 排序後相鄰元素比對：極速找出重複元素或最小相鄰差值（$O(N \log N)$ 取代 $O(N^2)$）

#### 1. 生活故事比喻：全班身高差距最接近的雙胞胎
如果全校有一千名學生，校長想要找出「哪兩位學生的身高差距最小（最接近的身高雙人組）」。
如果不排序，你必須拿第一位同學去跟其他 999 位同學逐一比身高，再拿第二位去跟剩下 998 位比……這要進行大約 $1,000 \times 1,000 / 2 = 500,000$（五十萬）次比對！
但如果校長先讓全體學生「依照身高由矮到高排成一長條縱隊」：
你想一想：全校身高差距最小的兩個人，有可能一個站在隊伍最前面、另一個站在隊伍最後面嗎？
**絕對不可能！**
全校最接近的兩個人，**必定在隊伍中緊緊挨在彼此的前後（必定是相鄰的兩個人）**！
因此，排好序之後，你只需要拿一把尺，沿著隊伍從頭走到尾，**只量測每一對「相鄰同學」的身高差**，只需 999 次比對，答案就呼之欲出了！

#### 2. 底層運作機制：複雜度的降維打擊
- **暴力法雙重迴圈**：  
  兩兩枚舉所有組合，時間複雜度為 $O(N^2)$。當 $N = 10^5$ 時，$N^2 = 10^{10}$，在 OJ 上直接超時掛掉。
- **排序 + 相鄰線性掃描**：  
  1. 排序花費：$O(N \log N)$。
  2. 單層迴圈掃描：$O(N)$。
  總時間複雜度：$O(N \log N) + O(N) = O(N \log N)$！
在電腦世界中，$10^5 \log_2(10^5) \approx 1.7 \times 10^6$ 次運算，不到 0.05 秒就能極速算出！

#### 3. 初學者常見陷阱：迴圈邊界引發 IndexError
當我們比較相鄰元素 `a[i]` 與 `a[i+1]` 時，迴圈範圍必須是 `range(len(a) - 1)`！
如果寫成 `range(len(a))`，最後一輪執行到 `a[i+1]` 時就會因為超出索引上限而暴斃引發 `IndexError`。

#### 4. APCS 實戰視野
「給定一數列，求任兩數之間最小絕對差值」是 APCS 實作觀念與二級實作的高頻考題。看到「任兩數差值最小」，反射神經必須立刻連結到「先排序，再掃描相鄰元素 `abs(a[i+1] - a[i])`」！

In [ ]:
# 範例 12.5.4：排序後相鄰掃描——極速尋找全數列最小差值
# 宣告一組雜亂的大數列
data = [105, 42, 88, 19, 73, 91, 55, 120, 89]
print("原始數列：", data)

# 關鍵步驟 1：升序排序
data.sort()
print("排序後數列：", data)

# 關鍵步驟 2：單層迴圈檢查相鄰兩數差值
min_diff = float("inf")  # 初始化為無窮大
best_pair = None

for i in range(len(data) - 1):
    diff = data[i + 1] - data[i]  # 由於已升序，後者必大於等於前者
    if diff < min_diff:
        min_diff = diff
        best_pair = (data[i], data[i + 1])

print(f"\n全數列差距最小的一對數字是: {best_pair}")
print(f"最小相鄰差值為: {min_diff}")

In [ ]:
# 填空題 12.5.4：找出相鄰最大的差距（最大跳躍）
# 任務：將數列排序後，找出相鄰兩數之間「最大的落差」。
milestones = [10, 45, 15, 80, 20]

# 請將數列排序
milestones.___()

max_gap = 0
# 請填入正確的迴圈終止邊界，避免 IndexError
for i in range(len(milestones) - ___):
    gap = milestones[i + 1] - milestones[i]
    if gap > max_gap:
        max_gap = gap

print("排序後里程碑：", milestones)
print("相鄰最大落差為：", max_gap)

In [ ]:
# ==========================================
# [4] Code 練習題 12.5.4
# 任務說明：
# 給定一組整數數列 nums。
# 請撰寫程式：
# 1. 將 nums 升序排序。
# 2. 找出任兩數之間差值最小的數值（最小差值）。
# 3. 印出排序後的數列與該最小差值。
#
# 【公開測試資料 1】
# nums = [30, 5, 12, 45, 16]
# 預期輸出：
# 排序後： [5, 12, 16, 30, 45]
# 最小差值： 4
#
# 【公開測試資料 2】
# nums = [100, 200, 105]
# 預期輸出：
# 排序後： [100, 105, 200]
# 最小差值： 5
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
nums = [30, 5, 12, 45, 16]
nums.sort()
print("排序後：", nums)
min_d = float("inf")
for i in range(len(nums) - 1):
    d = nums[i + 1] - nums[i]
    if d < min_d:
        min_d = d
print("最小差值：", min_d)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.5.4
# 任務說明：
# 某跑步比賽記錄了 6 位選手的完賽秒數：times = [85.2, 79.8, 92.1, 80.5, 74.0, 85.0]
# 請寫出一段程式，先對秒數排序，
# 接著找出成績差距最小的兩位選手秒數（最激烈的競爭對手），
# 印出這兩位選手的秒數與相差的秒數（保留小數一位）。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
times = [85.2, 79.8, 92.1, 80.5, 74.0, 85.0]
times.sort()
best_pair = None
min_gap = float("inf")
for i in range(len(times) - 1):
    gap = times[i + 1] - times[i]
    if gap < min_gap:
        min_gap = gap
        best_pair = (times[i], times[i + 1])
print("排序後秒數：", times)
print(f"競爭最激烈對手秒數: {best_pair[0]} 秒 vs {best_pair[1]} 秒，差距僅 {min_gap:.1f} 秒！")

### 12.5.5 已排序序列的雙指標夾擊（Two Pointers）：左右相向極速 $O(N)$ 解 Two Sum

#### 1. 生活故事比喻：相向而行的尋寶夾擊小隊
想像在一條筆直且標記著刻度（從小到大）的數線上，藏著一對神秘的寶石。任務要求你：「找出哪兩顆寶石的重量加起來剛好等於 100 公克」。
如果不排序，你必須在荒原上漫無目的地到處兩兩測試。
但如果寶石已經「由輕到重排成一長排」：
我們可以派出兩位探險員：
- 左隊員（`left`）：站在最左端（最輕的寶石處）。
- 右隊員（`right`）：站在最右端（最重的寶石處）。
兩人算一下眼前兩顆寶石的總和 `current_sum = weight[left] + weight[right]`：
- 如果加起來「太大了（`> 100`）」：說明太重了！這時候即使左隊員換成更重的也只會更大，唯一解法是——**右隊員往左退一步（`right -= 1`）**換輕一點的！
- 如果加起來「太小了（`< 100`）」：說明太輕了！**左隊員往右前進一步（`left += 1`）**換重一點的！
- 如果「剛好等於 100」：狂賀！當場尋寶成功！
兩位隊員相向而行，彼此絕不走回頭路，最多只要走 $N$ 步就能掃遍全場！

#### 2. 底層運作機制：雙指標演算法的單調性保證
這就是享譽全球演算法殿堂的**雙指標夾擊法（Two Pointers Technique）**！
其核心虛擬代碼：
```python
left = 0
right = len(a) - 1
while left < right:
    s = a[left] + a[right]
    if s == target:
        return (a[left], a[right])
    elif s < target:
        left += 1   # 總和不足，左指針右移增加總和
    else:
        right -= 1  # 總和超標，右指針左移減少總和
```
因為陣列已排序，每一步的增減方向是 100% 確定且絕對單調的。整個搜尋過程僅需 $O(N)$ 時間！

#### 3. 初學者常見陷阱：在未排序的串列上使用雙指標
雙指標夾擊法的唯一致命盲點：**「前提必須是已排序陣列！」**
如果串列本身是混亂的，右移不一定變大、左移不一定變小，指針的移動方向就會徹底失去數學邏輯，導致錯失正確解答。

#### 4. APCS 實戰視野
Two Sum（兩數之和）與 Three Sum（三數之和）是各類程式檢定中歷久彌新的常青樹考題。面對海量測資，暴力法 $O(N^2)$ 必定超時，而「先排序 $O(N \log N)$ ➔ 雙指標 $O(N)$」的黃金組合拳，是通往滿分必不可少的終極招式！

In [ ]:
# 範例 12.5.5：已排序串列的雙指標夾擊（Two Pointers）實戰
# 給定一組整數清單，尋找兩數之和等於 target 的組合
nums = [15, 3, 9, 2, 8, 12, 7, 20]
target = 23
print("原始數列：", nums, "，目標總和 target =", target)

# 關鍵前提：先排序！
nums.sort()
print("排序後數列：", nums)

# 初始化左右雙指標
left = 0
right = len(nums) - 1
found_pair = None

# 相向而行夾擊迴圈
while left < right:
    current_sum = nums[left] + nums[right]
    if current_sum == target:
        found_pair = (nums[left], nums[right])
        break
    elif current_sum < target:
        left += 1  # 總和太小，左指針往右移找更大的數
    else:
        right -= 1 # 總和太大，右指針往左移找更小的數

if found_pair:
    print(f"成功找到解！{found_pair[0]} + {found_pair[1]} = {target}")
else:
    print("找不到任何兩數之和等於目標！")

In [ ]:
# 填空題 12.5.5：雙指標夾擊模板填空
# 任務：在已排序的數列中，尋找兩數相加等於 50 的數對。
sorted_arr = [5, 12, 18, 25, 32, 38, 45]
target = 50

left = 0
right = len(sorted_arr) - 1

while left < right:
    cur = sorted_arr[left] + sorted_arr[right]
    if cur == target:
        print(f"找到組合：({sorted_arr[left]}, {sorted_arr[right]})")
        break
    elif cur < target:
        # 太小，左邊右移
        left += ___
    else:
        # 太大，右邊左移
        right -= ___

In [ ]:
# ==========================================
# [4] Code 練習題 12.5.5
# 任務說明：
# 給定一個整數串列 values 與目標整數 target。
# 請撰寫程式：
# 1. 將 values 排序。
# 2. 使用雙指標夾擊法找出相加恰好等於 target 的兩個數字。
# 3. 若找到印出 "找到：(較小數, 較大數)"，若找不到印出 "無解"。
#
# 【公開測試資料 1】
# values = [40, 10, 25, 5, 15]
# target = 35
# 預期輸出：
# 找到： (10, 25)
#
# 【公開測試資料 2】
# values = [1, 2, 3]
# target = 10
# 預期輸出：
# 無解
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
values = [40, 10, 25, 5, 15]
target = 35
values.sort()
l, r = 0, len(values) - 1
found = None
while l < r:
    s = values[l] + values[r]
    if s == target:
        found = (values[l], values[r])
        break
    elif s < target:
        l += 1
    else:
        r -= 1
if found:
    print(f"找到： ({found[0]}, {found[1]})")
else:
    print("無解")

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.5.5
# 任務說明：
# 某倉庫要裝載兩件貨物上同一輛載重卡車，卡車最大載重限制為 100 kg。
# 給定一批貨物重量：weights = [25, 35, 80, 15, 65, 45, 90]
# 請使用排序與雙指標夾擊法，找出「相加不超過 100 kg 的情況下，重量最重（最接近 100 kg）」的兩件貨物組合與其總重量。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
weights = [25, 35, 80, 15, 65, 45, 90]
weights.sort()
l, r = 0, len(weights) - 1
max_load = 0
best_combo = None

while l < r:
    s = weights[l] + weights[r]
    if s <= 100:
        if s > max_load:
            max_load = s
            best_combo = (weights[l], weights[r])
        l += 1  # 嘗試找更大的左側數值逼近
    else:
        r -= 1  # 超載，右側必須縮小

print(f"最佳兩件裝載組合: {best_combo}，合計載重: {max_load} kg")

### 12.5.6 競賽時空權衡：何時值得花 $O(N \log N)$ 排序來換取後續的高速線性處理？

#### 1. 生活故事比喻：磨刀不誤砍柴工
古人說：「磨刀不誤砍柴工」。
如果樵夫拿著一把鈍斧頭去砍整座森林的一千棵大樹，每砍一棵都要費盡九牛二虎之力（暴力 $O(N^2)$ 的低效痛苦）。
但如果樵夫在出發前，先花 30 分鐘坐在磨刀石旁把斧頭磨得無比鋒利（付出 $O(N \log N)$ 的前置排序代價），之後砍每一棵樹都像切豆腐一樣輕鬆寫意（後續 $O(N)$ 線性收割），總花費時間反而比不磨刀快了幾百倍！
這就是演算法設計中最經典的**「時空權衡與前置投資思維」**。

#### 2. 底層運作機制：複雜度的數學天平
在競賽中，我們該如何衡量「到底值不值得花時間排序」？
請看下方的運算次數對比（假設資料量 $N = 10^5$）：
- **不排序，直接暴力比對**：  
  $N^2 = (10^5)^2 = 10,000,000,000$（一百億次運算）。  
  以現代電腦每秒約 $10^8$ 次運算計算，需要整整 **100 秒**（保證 TLE 慘遭淘汰）！
- **先排序，再線性處理**：  
  排序代價：$N \log_2 N \approx 10^5 \times 17 \approx 1,700,000$ 次運算。  
  線性掃描：$N = 100,000$ 次運算。  
  總運算量：約 **180 萬次運算**，僅需 **0.02 秒** 即可瞬間通過！

#### 3. 初學者常見陷阱：忘記「原始索引」是否需要保留
排序雖然無比強大，但它有一個附帶影響：**會打亂資料最初輸入時的相對順序！**
如果題目最後要求你輸出：「這兩名同學在原始名冊上的學號或輸入序號是多少？」
如果你直接呼叫 `nums.sort()`，原始編號就被洗掉了！
解決之道是：在排序前，將資料打包為 `(數值, 原始索引)` 的元組，如 `[(val, idx) for idx, val in enumerate(nums)]`，再進行排序。這樣既享受了排序的高速，又絕不丟失原始編號！

#### 4. APCS 實戰視野
何時該果斷使用排序？記住三大訊號：
1. 資料量 $N \ge 10^4$，而題目看似要求兩兩比對。
2. 題目涉及區間、排程、極值、貪婪或兩數組合。
3. 查詢次數多次（一次排序，終身受用）。
掌握時空權衡，標誌著你已經脫離了新手語法期，真正具備了演算法架構師的宏觀戰略眼光！

In [ ]:
# 範例 12.5.6：帶原始索引排序技巧（保留最初輸入位置）
# 題目情境：找出相加等於 target 的兩數，但輸出必須是它們在「原始數列中的索引位置」！
raw_nums = [20, 7, 11, 15, 2]
target = 9
print("原始數列：", raw_nums)

# 技巧：使用 enumerate 將每個元素打包為 (數值, 原始索引)
indexed_data = [(val, i) for i, val in enumerate(raw_nums)]
print("打包原始索引後的資料：", indexed_data)

# 依數值（第 0 欄）排序
indexed_data.sort(key=lambda item: item[0])
print("依數值排序後：", indexed_data)

# 雙指標夾擊
left = 0
right = len(indexed_data) - 1
found_indices = None

while left < right:
    current_sum = indexed_data[left][0] + indexed_data[right][0]
    if current_sum == target:
        # 取出兩者的原始索引
        found_indices = (indexed_data[left][1], indexed_data[right][1])
        break
    elif current_sum < target:
        left += 1
    else:
        right -= 1

print(f"\n成功找到！相加為 {target} 的元素在原始數列中的索引為: {found_indices}")
print(f"驗證：raw_nums[{found_indices[0]}] + raw_nums[{found_indices[1]}] = "
      f"{raw_nums[found_indices[0]]} + {raw_nums[found_indices[1]]} = {target}")

In [ ]:
# 填空題 12.5.6：打包原始索引並排序
# 任務：將成績清單連同學號索引一起打包並由高到低（降序）排序。
scores = [65, 92, 78, 88]

# 請使用 enumerate 打包為 (分數, 學號索引)
packaged = [(s, idx) for idx, s in ___(scores)]

# 依分數降序排序
ranked = sorted(packaged, key=lambda item: item[0], reverse=___)

print("成績排行榜（含原始學號）：", ranked)
print(f"第一名是學號 {ranked[0][1]} 號，獲得 {ranked[0][0]} 分！")

In [ ]:
# ==========================================
# [4] Code 練習題 12.5.6
# 任務說明：
# 給定一組測量數據 measurements。
# 請撰寫程式：
# 1. 將每個數值與其原始索引打包：(數值, 索引)。
# 2. 依數值由小到大排序。
# 3. 輸出數值最小與最大的元素其「原始索引位置」。
#
# 【公開測試資料 1】
# measurements = [45, 12, 89, 34, 67]
# 預期輸出：
# 最小值 12 在原始索引: 1
# 最大值 89 在原始索引: 2
#
# 【公開測試資料 2】
# measurements = [100, 50]
# 預期輸出：
# 最小值 50 在原始索引: 1
# 最大值 100 在原始索引: 0
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
measurements = [45, 12, 89, 34, 67]
packed = [(val, i) for i, val in enumerate(measurements)]
packed.sort(key=lambda x: x[0])
print(f"最小值 {packed[0][0]} 在原始索引: {packed[0][1]}")
print(f"最大值 {packed[-1][0]} 在原始索引: {packed[-1][1]}")

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.5.6
# 任務說明：
# 某面試系統記錄了 5 位應徵者的筆試分數：scores = [72, 85, 90, 68, 85]
# 應徵者編號剛好是索引 0 到 4。
# 請將應徵者依筆試分數「由高到低（降序）」排序，若分數相同則依「原始編號由小到大（升序）」排序，
# 印出最終錄取順序中的應徵者編號名單。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
scores = [72, 85, 90, 68, 85]
candidates = [(s, i) for i, s in enumerate(scores)]
# 混合排序：分數降序 (-item[0])，編號升序 (item[1])
ranked_candidates = sorted(candidates, key=lambda c: (-c[0], c[1]))
order = [c[1] for c in ranked_candidates]
print("錄取順序（應徵者編號）：", order)

### 學習總結與通關回顧

恭喜你順利通關 **12.5 APCS 排序實戰應用：區間線段排序、貪婪前導與已排序雙指標**！

在本單元中，你完成了從「純語法操作」跨入「高階演算法架構」的重大蛻變：
- **「先排序，混亂立刻變規律」核心心法**：
  - 排序為無序世界注入單調性，將 $O(N^2)$ 暴力降維至 $O(N \log N)$。
- **區間線段起點排序**：
  - `intervals.sort(key=lambda x: x[0])`：由左至右掃描線，極速判定重疊與區間聯集覆蓋。
- **貪婪活動選擇原型**：
  - `activities.sort(key=lambda x: x[1])`：永遠挑選最早結束的活動，數學保證最多場次容納。
- **相鄰元素掃描術**：
  - 差值最小的元素排序後必定相鄰，單層迴圈掃遍全域。
- **已排序相向雙指標（Two Pointers）**：
  - 左右夾擊相向而行，線性 $O(N)$ 秒殺 Two Sum 兩數之和。
- **帶索引排序（Index Tagging）**：
  - `(val, idx)` 打包排序，兼得排序速度與原始編號完整性。

---
**下一關預告**：資料排好序之後，如果我們要找某個特定的數值，該怎麼找？為什麼普通的線性搜尋在龐大資料前會顯得力不從心？下一節 **12.6 線性搜尋（Linear Search）與安全查詢防護** 將帶你檢視搜尋演算法的原點與必備例外防禦！